## How to use this notebook

Run this first, top to bottom, from the repository root (or from `notebooks/`). It downloads the
raw dataset, cleans it, removes near-duplicates, and writes `data/processed/cleaned_recipes.parquet`.
Shared helpers now live in `src/preprocessing.py`; see its docstring for a shorter programmatic path.

# 01 - Preprocessing: FILIPINO_RECIPES_2K
Outputs: `data/processed/cleaned_recipes.parquet` (and later `alias_map.json`).

Run the cells in order. Cell 7 needs a manual decision before you continue.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Working directory:", Path.cwd())

## 0. Install (run once, then restart the kernel)

In [ ]:
import sys
print(sys.executable)
%pip install datasets pyarrow pandas rapidfuzz

## 1. Load the dataset and save the raw copy

In [ ]:
from datasets import load_dataset
import os

ds = load_dataset("joackimagno/FILIPINO_RECIPES_2K")
print(ds)

split = list(ds.keys())[0]
df = ds[split].to_pandas()

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/raw/raw_recipes.parquet", index=False)
print(df.shape)

## 2. Inspect structure and problems

In [ ]:
import pandas as pd
print(df.dtypes)
print("\nNulls:\n", df.isna().sum())
print("\nEmpty strings:\n", (df.astype(str).apply(lambda c: c.str.strip() == "")).sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate recipe names:", df["recipe_name"].duplicated().sum())

## 3. Read actual rows

In [ ]:
df.sample(3, random_state=1).T

## 4. Look at the ingredient fields closely

In [ ]:
for i in range(3):
    print(repr(df.loc[i, "ingredients"]))
    print(repr(df.loc[i, "ingredient_names"]))
    print(type(df.loc[i, "ingredient_names"]))
    print()

## 5. (Optional) Look at recipes that share a name

In [ ]:
dup_names = df[df["recipe_name"].duplicated(keep=False)].sort_values("recipe_name")
print(len(dup_names))
dup_names[["recipe_name", "total_time", "servings", "ingredients"]].head(20)

## 6. Clean and structure the data

This cell now calls `src/preprocessing.py`. The old inline cleaning code was removed; the module
is unit-tested and handles unicode, time parsing, and pipe-separated fields in one place.

In [ ]:
# Maintained cleaning lives in src/preprocessing.py (tested, pure functions).
from preprocessing import prepare_recipes, find_near_duplicate_pairs

clean, report = prepare_recipes(df)
pairs = find_near_duplicate_pairs(clean)
print("dropped for missing content:", report["dropped_empty"])
print("rows now:", len(clean))
print(len(pairs), "same-name pairs compared")

## 7a. Find candidate near-duplicate pairs (only among recipes with the same name)

`find_near_duplicate_pairs()` in `src/preprocessing.py` does this now.

In [ ]:
pairs.sort_values("ing_jaccard", ascending=False).head(40)

## 7b. Two tiers: auto-flag the clear copies, review the uncertain ones by eye

In [ ]:
tier1 = pairs[(pairs.ing_jaccard >= 0.8) & (pairs.time_diff <= 10)]
print(len(tier1), "pairs auto-flagged (tier 1)")

tier2 = pairs[(pairs.ing_jaccard < 0.8) & (pairs.ing_jaccard >= 0.4) &
              (pairs.instr_sim >= 0.75) & (pairs.time_diff <= 20)]
print(len(tier2), "pairs to review (tier 2)\n")

for _, r in tier2.iterrows():
    print(f"=== {r['name']} | rows {r['i']} vs {r['j']} | jac {r['ing_jaccard']} instr {r['instr_sim']} time_diff {r['time_diff']}")
    print("A:", clean.loc[r["i"], "ingredients"][:250])
    print("B:", clean.loc[r["j"], "ingredients"][:250], "\n")

## 7c. YOUR DECISION: list the tier 2 pairs that are true duplicates
Read the printout above. If A and B are the same ingredients in different wording, add `(i, j)` here. If they have different ingredients or amounts, they are real variants, so leave them out. When unsure, keep both.

In [ ]:
confirmed = [(527, 1003)]   # edit this list

## 7d. Group copies into clusters (union-find)

`duplicate_clusters()` handles this; you still decide which tier-2 pairs to confirm.

In [ ]:
# Union-find clustering is also in src/preprocessing.py; confirmed pairs stay manual.
from collections import Counter
from preprocessing import duplicate_clusters

clusters, to_drop, review = duplicate_clusters(pairs, clean, confirmed_pairs=confirmed)
print(len(clusters), "clusters,", sum(len(x) for x in clusters), "rows involved")
print("review-only pairs not merged:", len(review))

## 7e. Check clusters of size 3+ for wrongly chained recipes
A high worst-pair Jaccard (about 0.7+) means the whole cluster really is one recipe. A low one (below about 0.6) means it was chained through a middle row: put one of its row ids in `exclude_rows` to keep that whole cluster.

In [ ]:
from preprocessing import ingredient_tokens

def jac(a, b):
    A = ingredient_tokens(clean.loc[a, "ingredient_lines"])
    B = ingredient_tokens(clean.loc[b, "ingredient_lines"])
    return len(A & B) / max(1, len(A | B))

for cl in clusters:
    if len(cl) >= 3:
        worst = min(jac(a, b) for a, b in __import__("itertools").combinations(cl, 2))
        print(f"{clean.loc[cl[0], 'recipe_name'][:40]:40s} rows {cl}  worst pair jaccard {worst:.2f}")

In [ ]:
exclude_rows = []   # any cluster containing one of these row ids is left alone

## 7f. Drop all but one row per cluster (keep the one with the longest instructions)

In [ ]:
# Honour exclusions decided in the cell above, then drop one row per cluster.
if exclude_rows:
    cancelled = {r for cl in clusters if any(x in exclude_rows for x in cl) for r in cl}
    to_drop = [r for r in to_drop if r not in cancelled]

n0 = len(clean)
clean = clean.drop(index=to_drop)
print("removed near-duplicates:", n0 - len(clean))
print("rows now:", len(clean))
print("same name, different content (kept):", clean["recipe_name"].str.lower().duplicated().sum())

## 8. Final checks and save `cleaned_recipes.parquet`

In [ ]:
clean = clean.reset_index(drop=True)
clean.insert(0, "recipe_id", range(len(clean)))
print(clean.shape)
print(clean[["total_time", "total_time_min", "servings", "servings_min", "servings_max"]].sample(min(8, len(clean)), random_state=1))
print(clean["total_time_min"].isna().sum(), "rows with unparsed total time")

clean.to_parquet("data/processed/cleaned_recipes.parquet", index=False)
pd.read_parquet("data/processed/cleaned_recipes.parquet").dtypes

## 9. Ingredient vocabulary (starting point for the alias map)

In [ ]:
vocab = Counter(i for l in clean["ingredient_list"] for i in l)
print(len(vocab), "distinct ingredient names")
for term, n in vocab.most_common(80):
    print(f"{n:4d}  {term}")